In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
from IPython.display import display, Markdown
import os
from pathlib import Path
from agents.extensions.models.litellm_model import LitellmModel
import json

load_dotenv(override=True)
print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

OpenRouter API Key found: True


In [2]:
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
print(f"Tavily API Key found: {bool(os.environ.get('TAVILY_API_KEY'))}")

Tavily API Key found: True


In [3]:
import json
json.dumps({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})

'{"include_images": false, "include_answer": true, "search_depth": "advanced", "max_results": 10}'

In [4]:


# params = {
#       "command": "npx",
#       "args": [
#         "-y",
#         "mcp-remote",
#         f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}"
#       ],
#       "env": {
#         "DEFAULT_PARAMETERS": json.dumps({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})
#       }
# }

# tool_filter=create_static_tool_filter(blocked_tool_names=["tavily_research","tavily_map","tavily_crawl","tavily_skill"])

In [10]:
## http version
from agents.mcp import MCPServerStdio, MCPServerStreamableHttp, create_static_tool_filter
params = {
    "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}",
    "headers": {
        "DEFAULT_PARAMETERS": json.dumps({
            "include_images": False,
            "include_answer": "advanced",
            "max_results": 5,
            "search_depth": "advanced",
            "auto_parameters": True
        })
    }
}

tool_filter=create_static_tool_filter(blocked_tool_names=["tavily_research","tavily_map","tavily_crawl","tavily_skill"])

In [6]:
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server_python = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "ghcr.io/hrrodan/agent-workspace-mcp:latest" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

In [7]:
abs_tmp_dir

'/home/martin/projects/agents/6_mcp/tmp'

In [8]:
instructions = "You have tools available, use them if necessary."
request = "Check the docs from functools online and write a small example script in your workspace. Use the parameters include_answer: True, search_depth: advanced"
model = "openrouter/google/gemini-3-flash-preview"

In [11]:
async with MCPServerStreamableHttp(params=params, tool_filter=tool_filter, client_session_timeout_seconds=30) as mcp_server:
    async with mcp_server_python as mcp_server_python:
        agent = Agent(name="tool_manager", instructions=instructions, model=LitellmModel(model), mcp_servers=[mcp_server, mcp_server_python])
        with trace("tool_manager"):
            result = await Runner.run(agent, request, max_turns=30)
        display(Markdown(result.final_output))


/home/martin/projects/agents/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



I have researched the `functools` module documentation and created an example script demonstrating its most common and powerful tools: `lru_cache`, `wraps`, `partial`, and `singledispatch`.

### Small Example Script (`functools_example.py`)

```python
from functools import partial, lru_cache, wraps, singledispatch

# 1. @lru_cache: Memoization to speed up recursive calls
@lru_cache(maxsize=None)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# 2. @wraps: Preserve metadata of the original function when decorating
def simple_decorator(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        print(f"Calling function: {f.__name__}")
        return f(*args, **kwargs)
    return wrapper

@simple_decorator
def greet(name):
    """Greets the user."""
    return f"Hello, {name}!"

# 3. partial: Freeze some arguments of a function to create a new one
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)

# 4. @singledispatch: Function overloading based on the first argument's type
@singledispatch
def process(data):
    print(f"Generic processing: {data}")

@process.register(int)
def _(data):
    print(f"Processing an integer: {data + 10}")

@process.register(list)
def _(data):
    print(f"Processing a list of length: {len(data)}")

if __name__ == "__main__":
    print("--- lru_cache example ---")
    print(f"Fibonacci(30): {fibonacci(30)}")
    print(fibonacci.cache_info())

    print("\n--- wraps example ---")
    print(greet("Alice"))
    print(f"Function Name: {greet.__name__}") # Preserved by @wraps
    print(f"Docstring: {greet.__doc__}")      # Preserved by @wraps

    print("\n--- partial example ---")
    print(f"5 squared: {square(5)}")
    print(f"2 cubed: {cube(2)}")

    print("\n--- singledispatch example ---")
    process("Hello")  # Calls generic
    process(100)      # Calls int-specific
    process([1, 2, 3])# Calls list-specific
```

### Key Components Explained:
*   **`@lru_cache`**: Automatically caches the results of function calls. In the Fibonacci example, it turns an $O(2^n)$ operation into an $O(n)$ operation.
*   **`@wraps`**: Essential for writing decorators. It ensures that the wrapper function keeps the name and docstring of the function it is decorating.
*   **`partial`**: Allows you to create "specialized" versions of functions by pre-filling some of their arguments.
*   **`@singledispatch`**: Provides a way to achieve polymorphism in Python, allowing different code paths based on whether the input is an integer, a list, or another type.